# AI Assignment: A Classical Reasoning Agent

**Pipeline:** natural-language knowledge base → parser → facts + rules → forward chaining with unification → `True` / `False`

No generative model touches the answer. A regex parser turns the text into logic, and a forward-chaining engine (with unification and stratified negation-as-failure) decides entailment.

Running this notebook top to bottom will:
1. build and test the reasoner,
2. evaluate it on the public datasets (RuleTaker, CounterBench, FOLIO) and print accuracy and time,
3. write `submission/submission.csv`, `submission/report.pdf` and `submission/README.txt`.

If the test file is missing, the notebook falls back to a few built-in examples so you can see it working. Put the real test file at `TEST_PATH` (next cell) and re-run.

In [3]:
import re, time, csv, os, json
from collections import defaultdict

# ---- settings you may want to change ----
TEST_PATH = "test.csv"          # private test file: needs an id column, a context column and a question column
OUT_DIR   = "submission"        # everything required gets written here
CWA       = True                # closed-world assumption for "not" (overridden below if dev data says otherwise)

# Public datasets on Hugging Face. I could not check the exact repo names or column names
# offline, so if one fails to load the notebook just skips it. Edit these if needed.
PUBLIC_SOURCES = [
    ("tasksource/ruletaker", None, "train"),
    ("CounterBench/CounterBench", None, "train"),
    ("yale-nlp/FOLIO", None, "validation"),
]
MAX_PER_SOURCE = 3000
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Representation

* A **literal** is a tuple `(negated, predicate, args)`, e.g. `(False, "young", ("bob",))`.
* A **fact** is a ground literal.
* A **rule** is `body → head`, where the body is a list of literals. Terms starting with `?` are variables, e.g. `?X`.
* Words like *someone, something, they, it, people, things* become variables. Names like *Bob* or *bald eagle* become constants.

For example, `If someone is young and not kind then they are rude` becomes
`young(?X) ∧ ¬kind(?X) → rude(?X)`.

In [4]:
VAR_SUBJ = {"someone", "something", "they", "it", "he", "she", "people", "things", "anyone",
            "anything", "everyone", "everything", "person", "thing"}
VAR_OBJ  = {"someone", "something", "anyone", "anything"}
PRONOUN  = {"it", "they", "he", "she"}
GENERIC  = r"(?:things|people|persons|thing|person|animals|animal|ones)"

SEED_VERBS = set("""chase eat like need see visit love hate know want fear hug kiss help follow watch
call give take push pull hit kick ride play meet admire respect trust cause lead""".split())

def stem(w):
    return w[:-1] if w.endswith("s") and not w.endswith("ss") else w

def norm(s):
    s = s.strip().lower()
    s = re.sub(r"[.!?]+$", "", s).strip()
    for a, b in (("isn't", "is not"), ("aren't", "are not"), ("doesn't", "does not"), ("don't", "do not")):
        s = s.replace(a, b)
    s = re.sub(r"\b(the|a|an)\b\s*", "", s)
    return re.sub(r"\s+", " ", s).strip()

def split_and(txt):
    return [p.strip() for p in re.split(r"\s*,?\s*\band\b\s*", txt) if p.strip()]

def subj_term(s, pron):
    if s in PRONOUN:
        return pron
    if s in VAR_SUBJ:
        return "?X"
    return s.replace(" ", "_")

def obj_term(s, subj, pron):
    if s in PRONOUN:
        return pron
    if s in VAR_OBJ:
        return "?Y" if subj.startswith("?") else "?X"
    return s.replace(" ", "_")

def parse_literal(text, pron="?X"):
    # returns (neg, pred, args) or None when I can't make sense of the clause
    t = norm(text)
    m = re.match(r"^(.+?)\s+(?:is|are)\s+(not\s+)?(.+)$", t)
    if m:
        return (bool(m.group(2)), m.group(3).replace(" ", "_"), (subj_term(m.group(1), pron),))
    neg = False
    m = re.match(r"^(.+?)\s+(?:does not|do not)\s+(\w+)\s+(.+)$", t)
    if m:
        neg = True
        subj, verb, obj = m.group(1), stem(m.group(2)), m.group(3)
    else:
        toks = t.split()
        vi = next((i for i, w in enumerate(toks) if i > 0 and stem(w) in SEED_VERBS), None)
        if vi is None:   # unknown verb: guess the first word ending in "s"
            vi = next((i for i, w in enumerate(toks) if i > 0 and w.endswith("s") and not w.endswith("ss")), None)
        if vi is None or vi == len(toks) - 1:
            return None
        subj, verb, obj = " ".join(toks[:vi]), stem(toks[vi]), " ".join(toks[vi + 1:])
    s = subj_term(subj, pron)
    return (neg, verb, (s, obj_term(obj, s, pron)))

def is_ground(lit):
    return not any(a.startswith("?") for a in lit[2])

def parse_clauses(txt, pron):
    # "someone is smart and kind" -> the second clause has no subject of its own, so reuse the previous one
    out = []
    for piece in split_and(txt):
        lit = parse_literal(piece, pron)
        if lit is None and out and len(out[-1][2]) == 1:
            m = re.match(r"^(not\s+)?(.+)$", norm(piece))
            lit = (bool(m.group(1)), m.group(2).replace(" ", "_"), out[-1][2])
        out.append(lit)
        if lit is None:
            break
    return out

def parse_sentence(sent):
    # returns ("fact", lit) / ("rule", body, heads, text) / ("fail", text) / None for blank lines
    t = norm(sent)
    if not t:
        return None
    m = re.match(r"^if\s+(.+?),?\s+then\s+(.+)$", t)
    if m:
        body_txt, head_txt = m.groups()
        has_var = any(re.search(r"\b%s\b" % w, body_txt + " " + head_txt) for w in VAR_OBJ | {"people", "things"})
        first = split_and(body_txt)[0]
        # "it/they" refers to the variable if there is one, otherwise to the first named subject
        pron = "?X"
        if not has_var:
            lit0 = parse_literal(first)
            pron = lit0[2][0] if lit0 else "?X"
        body = parse_clauses(body_txt, pron)
        heads = parse_clauses(head_txt, pron)
        if None in body or None in heads:
            return ("fail", sent)
        return ("rule", body, heads, sent.strip())
    m = re.match(r"^(?:all\s+)?(.+?)\s+%s\s+(?:are|is)\s+(not\s+)?(.+)$" % GENERIC, t)
    if m:
        conds = [c.strip() for c in re.split(r"\s*,\s*(?:and\s+)?|\s+and\s+", m.group(1)) if c.strip()]
        body = [(False, c.replace(" ", "_"), ("?X",)) for c in conds]
        head = (bool(m.group(2)), m.group(3).replace(" ", "_"), ("?X",))
        return ("rule", body, [head], sent.strip())
    lit = parse_literal(t)
    if lit is None or not is_ground(lit):
        return ("fail", sent)
    return ("fact", lit)

def split_sentences(context):
    parts = []
    for line in re.split(r"\n+", context.strip()):
        parts += re.split(r"(?<=[.!?])\s+", line.strip())
    return [p for p in parts if p.strip()]

class Rule:
    def __init__(self, body, head, text):
        # positive literals first so the join binds variables before negation-as-failure is checked
        self.body = sorted(body, key=lambda l: l[0])
        self.head = head
        self.text = text

def build_kb(context):
    facts, rules, failed = [], [], []
    for s in split_sentences(context):
        r = parse_sentence(s)
        if r is None:
            continue
        if r[0] == "fact":
            facts.append(r[1])
        elif r[0] == "rule":
            for h in r[2]:
                rules.append(Rule(r[1], h, r[3]))
        else:
            failed.append(r[1])
    return facts, rules, failed

## 2. Reasoning engine

**Forward chaining.** Starting from the facts, I keep applying every rule whose body can be matched, until nothing new appears.

**Unification.** Matching a rule's body against known facts is a join. Each body literal is unified with the facts one by one, and the variable bindings are carried along (backtracking over alternatives).

**Negation.** Under the closed-world assumption (CWA), `not p` in a rule body means "p cannot be derived" (negation-as-failure). To keep this sound I stratify the rules: a predicate is fully computed before anything that depends on its negation runs. An explicit negative fact or rule head such as `not nice(bob)` is stored as a normal literal.

**Speed-ups.** (a) Rules that cannot lead to the query predicate are dropped before reasoning (a backward relevance pass). (b) Facts are indexed by `(sign, predicate)`.

**Traces.** Every derived fact remembers the rule and premises that produced it, so a proof tree can be printed.

In [5]:
def unify(pat, vals, theta):
    if len(pat) != len(vals):
        return None
    t, copied = theta, False
    for p, v in zip(pat, vals):
        if p[0] == "?":
            if p in t:
                if t[p] != v:
                    return None
            else:
                if not copied:
                    t, copied = dict(t), True
                t[p] = v
        elif p != v:
            return None
    return t

def subst(args, theta):
    return tuple(theta.get(a, a) if a[0] == "?" else a for a in args)

def join(body, i, theta, used, idx, cwa):
    if i == len(body):
        yield theta, used
        return
    neg, pred, args = body[i]
    if neg and cwa:
        # negation as failure: succeeds only if no positive fact matches
        for fargs in idx.get((False, pred), ()):
            if unify(args, fargs, theta) is not None:
                return
        yield from join(body, i + 1, theta, used + [("naf", pred, subst(args, theta))], idx, cwa)
        return
    for fargs in list(idx.get((neg, pred), ())):
        t2 = unify(args, fargs, theta)
        if t2 is not None:
            yield from join(body, i + 1, t2, used + [(neg, pred, fargs)], idx, cwa)

def relevant_rules(rules, goal_pred):
    needed, changed = {goal_pred}, True
    while changed:
        changed = False
        for r in rules:
            if r.head[1] in needed:
                for _, p, _ in r.body:
                    if p not in needed:
                        needed.add(p); changed = True
    return [r for r in rules if r.head[1] in needed]

def stratify(rules, cwa):
    st = defaultdict(int)
    limit = len(rules) + 2
    for _ in range(limit * 2):
        changed = False
        for r in rules:
            for neg, p, _ in r.body:
                need = st[p] + (1 if (neg and cwa) else 0)
                if st[r.head[1]] < need:
                    st[r.head[1]] = need; changed = True
        if not changed:
            return {id(r): st[r.head[1]] for r in rules}
    return {id(r): 0 for r in rules}      # negation loop: just treat everything as one layer

def forward_chain(facts, rules, cwa=True):
    prov, idx = {}, defaultdict(set)
    def add(f, why):
        if f in prov:
            return False
        prov[f] = why
        idx[(f[0], f[1])].add(f[2])
        return True
    for f in facts:
        add(f, ("given",))
    layer = stratify(rules, cwa)
    for level in sorted(set(layer.values())):
        todo = [r for r in rules if layer[id(r)] == level]
        changed = True
        while changed:
            changed = False
            for r in todo:
                for theta, used in list(join(r.body, 0, {}, [], idx, cwa)):
                    h = subst(r.head[2], theta)
                    if any(a[0] == "?" for a in h):
                        continue          # unsafe rule, head variable never got bound
                    if add((r.head[0], r.head[1], h), ("rule", r.text, used)):
                        changed = True
    return prov

def fmt(l):
    if l[0] == "naf":
        return "not %s(%s) [cannot be derived]" % (l[1], ",".join(l[2]))
    return "%s%s(%s)" % ("not " if l[0] else "", l[1], ",".join(l[2]))

def explain(fact, prov, depth=0, seen=None):
    seen = seen if seen is not None else set()
    pad = "  " * depth
    why = prov.get(fact)
    if why is None:
        return [pad + fmt(fact) + "   <- not derivable"]
    if why[0] == "given" or fact in seen:
        return [pad + fmt(fact) + "   [given]"]
    seen.add(fact)
    lines = [pad + fmt(fact) + "   <- " + why[1]]
    for p in why[2]:
        lines += explain(p, prov, depth + 1, seen) if p[0] != "naf" else [pad + "  " + fmt(p)]
    return lines

def entails(context, query, cwa=True, trace=False):
    facts, rules, _ = build_kb(context)
    q = parse_literal(query)
    if q is None or not is_ground(q):
        return (False, ["could not parse the query"]) if trace else False
    prov = forward_chain(facts, relevant_rules(rules, q[1]), cwa)
    pos = (False, q[1], q[2])
    if q in prov:
        ans, why = True, explain(q, prov)
    elif q[0] and cwa and pos not in prov:
        ans, why = True, ["not %s(%s) holds because it cannot be derived (closed-world assumption)" % (q[1], ",".join(q[2]))]
    else:
        ans, why = False, ["%s is not derivable from the knowledge base" % fmt(q)]
    return (ans, why) if trace else ans

## 3. Quick sanity checks (including the example from the assignment sheet)

In [6]:
DEMO = [
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.", True),
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is not smart.", False),
 ("The cat chases the dog. The dog is big. If something chases the dog then it is angry.", "The cat is angry.", True),
 ("Alan is big. Alan is red. Big, red things are nice.", "Alan is nice.", True),
 ("Alan is big. Big, red things are nice.", "Alan is nice.", False),
 ("Fiona is young. Erin is young. Erin is kind. If someone is young and not kind then they are rude.", "Fiona is rude.", True),
 ("Fiona is young. Erin is young. Erin is kind. If someone is young and not kind then they are rude.", "Erin is rude.", False),
 ("Bob is rough. If someone is rough then they are not nice.", "Bob is not nice.", True),
 ("The bear is big. If the bear is big then the bear is red. If the bear is red then the bear needs the cat.", "The bear needs the cat.", True),
 ("The bear is cold. If the bear is cold and it is nice then it is green.", "The bear is green.", False),
]
for i, (c, q, y) in enumerate(DEMO):
    got = entails(c, q, cwa=True)
    print("ok " if got == y else "BAD", i, q, "->", got)

ans, why = entails(DEMO[0][0], DEMO[0][1], trace=True)
print("\nReasoning trace for the assignment example (answer = %s):" % ans)
print("\n".join(why))

ok  0 Bob is successful. -> True
ok  1 Bob is not smart. -> False
ok  2 The cat is angry. -> True
ok  3 Alan is nice. -> True
ok  4 Alan is nice. -> False
ok  5 Fiona is rude. -> True
ok  6 Erin is rude. -> False
ok  7 Bob is not nice. -> True
ok  8 The bear needs the cat. -> True
ok  9 The bear is green. -> False

Reasoning trace for the assignment example (answer = True):
successful(bob)   <- If someone is smart and kind, then they are successful.
  smart(bob)   <- If someone is young, then they are smart.
    young(bob)   [given]
  kind(bob)   [given]


## 4. Loading the public data

The loader looks for the usual column names (`context`/`premises`/`theory`, `question`/`statement`/`conclusion`, `label`/`answer`) because the three datasets name things differently. Labels `True`/`entailment` map to `True`, everything else (False, Uncertain, not entailment) maps to `False`, since the task is binary.

In [7]:
def pick(d, names):
    for n in names:
        if n in d and d[n] is not None:
            return d[n]
    return None

def to_bool(x):
    return str(x).strip().lower() in {"true", "entailment", "entailed", "yes", "1", "t"}

def as_text(x):
    return " ".join(x) if isinstance(x, (list, tuple)) else str(x)

def normalise_row(d, i):
    ctx = pick(d, ["context", "premises", "theory", "story", "facts", "text"])
    qs  = pick(d, ["question", "statement", "conclusion", "query", "hypothesis"])
    lab = pick(d, ["label", "answer", "flag", "gold"])
    if ctx is None or qs is None:
        return None
    return {"id": pick(d, ["id", "ID", "idx"]) if pick(d, ["id", "ID", "idx"]) is not None else i,
            "context": as_text(ctx), "question": as_text(qs),
            "label": None if lab is None else to_bool(lab)}

def load_public():
    data = {}
    try:
        from datasets import load_dataset
    except Exception as e:
        print("`datasets` library not available (%s); using the built-in demo set." % type(e).__name__)
        return {"demo": [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]}
    for name, cfg, split in PUBLIC_SOURCES:
        try:
            ds = load_dataset(name, cfg, split=split) if cfg else load_dataset(name, split=split)
            rows = [normalise_row(dict(r), i) for i, r in enumerate(ds)]
            rows = [r for r in rows if r and r["label"] is not None][:MAX_PER_SOURCE]
            data[name] = rows
            print("loaded %-28s %d examples" % (name, len(rows)))
        except Exception as e:
            print("skipped %-27s (%s)" % (name, str(e)[:80]))
    return data or {"demo": [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]}

public = load_public()

`datasets` library not available (ModuleNotFoundError); using the built-in demo set.


## 5. Experiments on the public data (accuracy and time), CWA vs. open-world

In [8]:
def evaluate(rows, cwa):
    t0 = time.perf_counter()
    hits, parse_fail = 0, 0
    for r in rows:
        try:
            pred = entails(r["context"], r["question"], cwa=cwa)
        except Exception:
            pred, parse_fail = False, parse_fail + 1
        hits += (pred == r["label"])
    dt = time.perf_counter() - t0
    return 100.0 * hits / max(1, len(rows)), dt, parse_fail

results = {}
for name, rows in public.items():
    for cwa in (True, False):
        acc, dt, pf = evaluate(rows, cwa)
        results[(name, cwa)] = (acc, dt, len(rows), pf)
        print("%-28s CWA=%-5s accuracy=%6.2f%%  time=%7.3fs  (%d queries, %d crashes)" % (name, cwa, acc, dt, len(rows), pf))

# pick the setting with the better average accuracy over the public data
def avg(cwa):
    v = [results[(n, cwa)][0] for n in public]
    return sum(v) / len(v)
CWA = avg(True) >= avg(False)
print("\nChosen setting: CWA =", CWA)

demo                         CWA=True  accuracy=100.00%  time=  0.001s  (10 queries, 0 crashes)
demo                         CWA=False accuracy= 90.00%  time=  0.000s  (10 queries, 0 crashes)

Chosen setting: CWA = True


## 6. Run on the test set and write `submission.csv`

In [9]:
def read_test(path):
    import pandas as pd
    df = pd.read_csv(path) if path.endswith(".csv") else pd.read_json(path)
    rows = []
    for i, rec in enumerate(df.to_dict("records")):
        r = normalise_row(rec, i)
        if r:
            rows.append(r)
    return rows

if os.path.exists(TEST_PATH):
    test_rows = read_test(TEST_PATH)
    print("Test file found:", TEST_PATH, "-", len(test_rows), "queries")
else:
    print("!! %s not found. Writing a submission for the demo examples only, so replace this with the real test file." % TEST_PATH)
    test_rows = [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]

t0 = time.perf_counter()
preds = []
for r in test_rows:
    try:
        preds.append(bool(entails(r["context"], r["question"], cwa=CWA)))
    except Exception:
        preds.append(False)
infer_time = time.perf_counter() - t0

with open(os.path.join(OUT_DIR, "submission.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])
    for r, p in zip(test_rows, preds):
        w.writerow([r["id"], p])

print("Wrote %s/submission.csv (%d rows)" % (OUT_DIR, len(preds)))
print("Total inference time: %.3f s (%.2f ms per query)" % (infer_time, 1000 * infer_time / max(1, len(preds))))

# Final headline numbers on the public data with the chosen setting
tot = sum(results[(n, CWA)][2] for n in public)
final_acc = sum(results[(n, CWA)][0] * results[(n, CWA)][2] for n in public) / max(1, tot)
final_time = sum(results[(n, CWA)][1] for n in public)
print("Public-data accuracy: %.2f%%   public-data time: %.3f s" % (final_acc, final_time))

!! test.csv not found. Writing a submission for the demo examples only, so replace this with the real test file.
Wrote submission/submission.csv (10 rows)
Total inference time: 0.001 s (0.05 ms per query)
Public-data accuracy: 100.00%   public-data time: 0.001 s


## 7. Report and README

These cells write `report.pdf` and `README.txt` into the submission folder. The experiment numbers come from the run above, so re-run everything before submitting.

In [10]:
def trace_text(i):
    return "\n".join(entails(DEMO[i][0], DEMO[i][1], cwa=True, trace=True)[1])

exp_lines = []
for (name, cwa), (acc, dt, n, pf) in results.items():
    exp_lines.append("%s | CWA=%s | %d queries | accuracy %.2f%% | %.3f s" % (name, cwa, n, acc, dt))

REPORT = [
 ("1. Representation",
  "Every sentence is converted to first-order Horn-style logic. A literal is (negated, predicate, arguments). Facts are ground "
  "literals, and rules are body -> head, where the body is a conjunction of literals. Pronouns and words such as someone, something, "
  "people and things become variables (?X, ?Y), while names become constants. The parser is regex-based and handles: simple facts "
  "(Bob is young), negated facts (Bob is not kind), relations (The cat chases the dog), if-then rules with conjunctions, "
  "and generic rules such as 'Big, red things are nice'. Rule heads can be negative literals, and sentences the parser cannot read "
  "are skipped and counted instead of crashing the system."),
 ("2. Reasoning algorithms",
  "The decision is made by forward chaining. Rule bodies are matched against the fact base by unification, with backtracking "
  "over variable bindings. Negation as failure (closed-world assumption) is supported through stratification, so a predicate is "
  "fully derived before any rule that negates it is applied. Facts are indexed by (sign, predicate). Before chaining starts, "
  "rules that cannot contribute to the query predicate are removed by a backward relevance analysis. Each derived fact stores the "
  "rule and premises that produced it, which gives proof traces for free. A query is True if the literal is derived; "
  "a negated query is also True when the positive atom cannot be derived (CWA). Everything else is False."),
 ("3. Experiments on the public data",
  "\n".join(exp_lines) + "\nI compared the closed-world and the open-world reading of 'not' and kept the one with higher "
  "average accuracy (CWA = %s). Final public accuracy: %.2f%%, total time %.3f s. Limitations: the parser is rule-based, so free-form "
  "FOLIO sentences with quantifiers, disjunction or unusual grammar are only partly covered, and CounterBench causal phrasing "
  "is handled only when it reduces to if-then sentences. I did not look at any private test data." % (CWA, final_acc, final_time)),
 ("4. Example reasoning traces",
  "Example A (from the assignment sheet): Bob is young, Bob is kind; young -> smart; smart and kind -> successful. Query: Bob is successful.\n"
  + trace_text(0) +
  "\n\nExample B (negation as failure): Fiona is young; Erin is young and kind; young and not kind -> rude. Query: Fiona is rude.\n"
  + trace_text(5)),
]

README = """README
======
Name: Patani Kalp Bharatkumar
Roll no: M26CSA009

Files
  reasoning_agent.ipynb : full implementation (parser, forward-chaining reasoner, evaluation)
  submission.csv        : predictions (id,label)
  report.pdf            : short report (representation, algorithms, experiments, traces)

How to run
  1. pip install datasets pandas reportlab
  2. Put the test file next to the notebook as test.csv (columns: id, context, question)
     or change TEST_PATH in the first cell.
  3. Run all cells. The last cells print accuracy and inference time and write the three
     files above into the submission/ folder.

Method in one line
  Regex parser -> facts + rules -> forward chaining with unification and stratified
  negation-as-failure -> True/False. No generative AI is used for the prediction.
"""

with open(os.path.join(OUT_DIR, "README.txt"), "w") as f:
    f.write(README)

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
    ss = getSampleStyleSheet()
    mono = ParagraphStyle("mono", parent=ss["Code"], fontSize=7.5, leading=9)
    story = [Paragraph("AI Assignment: Classical Reasoning Agent", ss["Title"]), Paragraph("Patani Kalp Bharatkumar | M26CSA009", ss["Normal"]), Spacer(1, 8)]
    def esc(s): return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    for head, text in REPORT:
        story.append(Paragraph(head, ss["Heading2"]))
        parts = text.split("\n")
        for ln in parts:
            if ln.startswith(" ") or "(" in ln and "<-" in ln:
                story.append(Preformatted(ln, mono))
            elif ln.strip():
                story.append(Paragraph(esc(ln), ss["BodyText"]))
        story.append(Spacer(1, 6))
    SimpleDocTemplate(os.path.join(OUT_DIR, "report.pdf"), pagesize=A4).build(story)
    print("Wrote report.pdf and README.txt in", OUT_DIR)
except Exception as e:
    with open(os.path.join(OUT_DIR, "report.txt"), "w") as f:
        f.write("\n\n".join(h + "\n" + t for h, t in REPORT))
    print("reportlab missing (%s), wrote report.txt instead. Install reportlab and re-run for the PDF." % e)

Wrote report.pdf and README.txt in submission
